# Evaluate & Promote

Full evaluation of the stacked ensemble on the held-out test set, one
antisymmetric evidence observation per physical match. The candidate is
loaded from the manifest (never a latest-run lookup).

Promotion is a probability-first gate: the candidate replaces production
(`ensemble_lr_model` @champion) when its test log loss is strictly lower than
the incumbent's AND its ROC-AUC is within `PROMOTION_TOLERANCE`, OR its
ROC-AUC is strictly higher AND its log loss is within `PROMOTION_TOLERANCE`
(0.01) on the same `test_evidence` matrix. First promotion happens when no
champion exists.

A report section always runs (ROC/PR/calibration, confusion matrix,
best-effort SHAP) regardless of the promotion outcome. Deployment is decoupled
from promotion — see `src/flows/deploy.py`.


In [ ]:
from pathlib import Path
from src.constants import (
    MODELS_ARTIFACTS,
    CANDIDATE_MANIFEST,
    CHAMPION_ALIAS,
    DATA_PROCESSED,
    PRODUCTION_MODEL,
    STACK_ORDER,
    load_env,
)

input_dir = str(DATA_PROCESSED)
output_dir = str(MODELS_ARTIFACTS)
Path(output_dir).mkdir(parents=True, exist_ok=True)
random_state = 42
production_model_name = PRODUCTION_MODEL  # MLflow registered model name
candidate_manifest = str(CANDIDATE_MANIFEST)  # written by 04
shap_sample_size = 1500  # SHAP subsample cap (report-only, wrapped in try/except)
calibration_temp_bounds = (
    0.05,
    20.0,
)  # temperature-fit bounds for the OOF-calibrated stacker (papermill-overridable)
calibration_brier_guard_tolerance = 0.01  # Brier may degrade at most this much vs raw (guardrail)
force_promote = False  # --force-promote: bypass the metric gate and always promote
rebuild_cmd = ""  # optional manual deploy hook, off by default (never wired to a build)
report_dpi = 150
report_figsize = (8, 6)
calibration_n_bins = 10

load_env()

In [ ]:
import hashlib
import json

import subprocess
from datetime import UTC, date, datetime
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking.client import MlflowClient


from src.constants import (
    CALIBRATION_HASH_TAG,
    CALIBRATION_STATE,
    CALIBRATION_URI_TAG,
    CHAMPION_CURVE_ARTIFACT,
    CHAMPION_CURVE_HASH_TAG,
    CHAMPION_CURVE_URI_TAG,
    EVAL_MAX_DATE_KEY,
    EVAL_SPLIT_SIZE_KEY,
    METRIC_PREFIX,
    MIN_TRAINING_DATE_KEY,
    TRAIN_FRACTION_KEY,
    VAL_FRACTION_KEY,
    TEST_FRACTION_KEY,
    CALIBRATION_PLOTS,
    PROMOTION_TOLERANCE,
    TRAIN_DATA_MAX_DATE_KEY,
    RECENCY_HALF_LIFE_DAYS,
    RECENCY_HALF_LIFE_KEY,
    RECENCY_CUTOFF_KEY,
    build_lineage_tags,
    FEATURE_COLS_HASH_TAG,
    FEATURE_COLS_TAG,
    normalize_gbdt_framework,
    normalize_linear_framework,
)

from src.evaluate.promotion import (
    METRIC_NAMES,
    check_candidate_cutoff,
    compute_metrics,
    decide_promotion,
    read_champion_metrics,
    resolve_champion,
    resolve_champion_feature_contract,
)

from src.evaluate.symmetry import evidence_to_probability
from src.evaluate.calibration import (
    apply_temperature,
    expected_calibration_error,
    select_temperature,
)

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
)

from sklearn.calibration import calibration_curve

import matplotlib.pyplot as plt
from IPython.display import display

import seaborn as sns

In [ ]:
# ── Load test data (one antisymmetric evidence row per physical match) ──
# The stack head evaluates on the antisymmetric base evidence from 03
# (test_evidence.parquet + test_eval.parquet, one row per match) — never
# X_test or raw directional mirrors.
test_evidence = pd.read_parquet(f"{output_dir}/test_evidence.parquet")
test_eval = pd.read_parquet(f"{output_dir}/test_eval.parquet")
info_test = pd.read_parquet(f"{input_dir}/info_test.parquet").reset_index(drop=True)

if tuple(test_evidence.columns) != STACK_ORDER:
    raise ValueError(
        f"test_evidence columns must be exactly {STACK_ORDER}, got {list(test_evidence.columns)}"
    )
if list(test_eval.columns) != ["match_id", "match_won"]:
    raise ValueError(
        f"test_eval columns must be [match_id, match_won], got {list(test_eval.columns)}"
    )
if not test_eval["match_id"].is_unique:
    raise ValueError("test_eval must hold exactly one row per physical match")
y_eval = test_eval["match_won"].reset_index(drop=True)

# One info_test row per physical match in the chosen orientation (03's
# deterministic convention: the row whose player_id is lexicographically
# smaller), ordered to match test_eval. Used for incumbent contexts and the
# surface error analysis below.
chosen_info = (
    info_test[
        np.asarray(info_test["player_id"], dtype=str)
        <= np.asarray(info_test["opponent_id"], dtype=str)
    ]
    .set_index("match_id")
    .loc[test_eval["match_id"]]
    .reset_index()
)
assert (chosen_info["match_id"] == test_eval["match_id"]).all()
print(f"Test set: {len(test_eval)} matches, evidence cols: {list(test_evidence.columns)}")

In [ ]:
# ── Load candidate from the manifest written by 04 ──
# Exact run handoff: candidate_run_id + model_uri from disk. No latest lookup.
with open(candidate_manifest) as f:
    manifest = json.load(f)
print(f"Candidate run:  {manifest['candidate_run_id']}")
print(f"Model URI:      {manifest['model_uri']}")
candidate = mlflow.sklearn.load_model(manifest["model_uri"])
assert candidate is not None
print("Loaded candidate meta-model")

In [ ]:
# ── Compute candidate predictions on the evidence matrix (one row per match) ──
assert candidate is not None
# Raw stacker probabilities. The calibrated y_proba (temperature applied
# with the OOF-fitted t_cal) is computed in the temperature cell below,
# once t_cal exists — y_proba_raw is kept for diagnostics and plots.
y_proba_raw = candidate.predict_proba(test_evidence[list(STACK_ORDER)].to_numpy())[:, 1]

# Base-model comparison: each base's symmetric probability sigmoid(evidence),
# one observation per physical match.
print("Base model test ROC-AUC / Accuracy (symmetric evidence):")
for name in STACK_ORDER:
    p = evidence_to_probability(test_evidence[name])
    auc = roc_auc_score(y_eval, p)
    acc = accuracy_score(y_eval, np.asarray(p >= 0.5, dtype=int))
    print(f"  {name:8s} AUC {auc:.4f}  Acc {acc:.4f}")

In [ ]:
# ── Resolve the incumbent's recorded metrics from the champion's version tags ──

client = MlflowClient()

champion = resolve_champion(client)
prod_metrics = None

if champion is not None:
    prod_metrics = read_champion_metrics(client, champion)
    print(
        f"Incumbent metrics from champion {production_model_name} v{champion.version} "
        f"run {champion.run_id}: " + ", ".join(f"{m}={prod_metrics[m]:.4f}" for m in METRIC_NAMES)
    )
else:
    print("No production model found — first promotion.")

In [ ]:
# ── Fit temperature on strictly forward stack-CV evidence ──
# Invariant: t_cal is fitted ONLY on the strictly forward (causal) stack-CV
# evidence from 03 (train_val_stack_cv.parquet) plus its per-match labels —
# the held-out test set (test_eval / y_eval) is NEVER used to fit t_cal, only
# to evaluate the fixed temperature. The artifact already carries match_date,
# fold, and match_won, so no external train-label lookup is required.
stack_cv = pd.read_parquet(f"{output_dir}/train_val_stack_cv.parquet")

# Per-match labels and folds are persisted on the artifact. Place the
# predictions in chronological (match_date) order for the walk-forward
# selection below (which fits each fold only on earlier folds).
cal_frame = (
    stack_cv.assign(match_id=stack_cv["match_id"].astype(str))
    .sort_values("match_date", kind="stable")
    .reset_index(drop=True)
)
print(f"Strictly forward stack-CV predictions: {len(cal_frame)} matches")
print(cal_frame["fold"].value_counts().sort_index().to_string())

calibration = select_temperature(
    cal_frame["stack_pred_cv"],
    cal_frame["match_won"],
    cal_frame["fold"],
    bounds=calibration_temp_bounds,
    brier_guard_tolerance=calibration_brier_guard_tolerance,
)
t_cal = calibration.temperature
print(
    f"Calibration selection ({'ACCEPTED' if calibration.accepted else 'REJECTED'}): "
    f"fitted t={calibration.fitted_temperature:.4f}, selected t={t_cal:.4f}"
)
print(
    f"  walk-forward OOF log loss: raw {calibration.raw_log_loss:.4f} -> "
    f"calibrated {calibration.calibrated_log_loss:.4f}"
)
print(
    f"  walk-forward OOF Brier:    raw {calibration.raw_brier:.4f} -> "
    f"calibrated {calibration.calibrated_brier:.4f}"
)

# Persist only a selection-approved temperature; deploy re-downloads and
# hash-verifies these exact bytes.
CALIBRATION_STATE.parent.mkdir(parents=True, exist_ok=True)
CALIBRATION_STATE.write_bytes(json.dumps({"temperature": float(t_cal)}).encode() + b"\n")
print(f"Persisted calibration state: {CALIBRATION_STATE}")

# Apply the selection-approved temperature to the final test probabilities.
# A rejected calibration uses t=1.0, exactly the uncalibrated candidate.
y_proba = np.asarray(apply_temperature(y_proba_raw, t_cal))
y_pred = np.asarray(y_proba >= 0.5, dtype=int)
print(f"Selected candidate test probabilities use t_cal={t_cal:.4f} (0.5 threshold unchanged)")

# Reliability diagnostics on the test set (display-only; never gates promotion).
print("\nReliability diagnostics (test set, display-only):")
print(
    f"  ECE (n_bins={calibration_n_bins}): raw {expected_calibration_error(y_eval, y_proba_raw, n_bins=calibration_n_bins):.4f} "
    f"vs calibrated {expected_calibration_error(y_eval, y_proba, n_bins=calibration_n_bins):.4f}"
)
reli = pd.DataFrame({"y_proba": y_proba, "y_eval": y_eval.to_numpy()})
reli["bin"] = pd.cut(reli["y_proba"], bins=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True)
binned = (
    reli.groupby("bin", observed=True)
    .agg(
        predicted_mean=("y_proba", "mean"),
        actual_fraction=("y_eval", "mean"),
        n=("y_eval", "size"),
    )
    .to_string()
)
print(binned)

## Metrics — candidate vs champion

Candidate gate metrics (log_loss, roc_auc, accuracy, brier) are computed on
`test_evidence`. The incumbent's gate metrics come from the metric tags
recorded on the `@champion` version at promotion time (read via MlflowClient),
not from any live service.

Calibration uses walk-forward CV across the chronologically ordered strictly-forward stack-CV folds
(each temperature fits only earlier folds, scores the next) and is retained
only when it strictly improves both log loss and Brier. The test set never
selects calibration; the promotion gate compares the selected candidate
against the incumbent's recorded metrics on identical evidence.


In [ ]:
# ── Metrics ──

# Candidate-only headline numbers (kept for continuity).

print("Test Set Performance (candidate)")

print(f"ROC-AUC:    {roc_auc_score(y_eval, y_proba):.4f}")
print(f"Accuracy:   {accuracy_score(y_eval, y_pred):.4f}")
print(f"Precision:  {precision_score(y_eval, y_pred):.4f}")
print(f"Recall:     {recall_score(y_eval, y_pred):.4f}")
print(f"F1:         {f1_score(y_eval, y_pred):.4f}")

print()
print(classification_report(y_eval, y_pred))


# First compare the selected candidate with its uncalibrated form as a test
# diagnostic only. The selection above already happened on later OOF rows,
# never on this test set. prod_metrics are the champion's recorded gate
# metrics, so the promotion comparison stays separate and like-for-like.

raw_pred = np.asarray(y_proba_raw >= 0.5, dtype=int)
raw_cand_metrics = compute_metrics(y_eval, y_proba_raw, raw_pred)
cand_metrics = compute_metrics(y_eval, y_proba, y_pred)

print(f"\n{'metric':10s} {'raw candidate':>14s} {'selected candidate':>18s} {'delta':>10s}")
for m in METRIC_NAMES:
    print(
        f"{m:10s} {raw_cand_metrics[m]:14.4f} {cand_metrics[m]:18.4f} {cand_metrics[m] - raw_cand_metrics[m]:+10.4f}"
    )

print(f"\n{'metric':10s} {'selected candidate':>18s} {'champion (recorded)':>20s} {'delta':>10s}")

for m in METRIC_NAMES:
    if prod_metrics is not None:
        print(
            f"{m:10s} {cand_metrics[m]:18.4f} {prod_metrics[m]:20.4f} "
            f"{cand_metrics[m] - prod_metrics[m]:+10.4f}"
        )

    else:
        print(f"{m:10s} {cand_metrics[m]:18.4f} {'n/a':>20s} {'n/a':>10s}")

In [ ]:
# ── Error analysis by surface (chosen orientation, one observation per match) ──
if "surface" in info_test.columns:
    chosen_info["correct"] = (y_pred == y_eval).to_numpy()
    print("Accuracy by surface:")
    print(chosen_info.groupby("surface")["correct"].mean().to_string())

## Promotion decision

Promote iff the candidate's test log loss is strictly lower than the
incumbent's AND its ROC-AUC is within `PROMOTION_TOLERANCE`, OR its ROC-AUC is
strictly higher AND its log loss is within `PROMOTION_TOLERANCE` (0.01), or
when no production exists yet (first promotion). Idempotency guard: if
`@champion` already points at this run's version, no re-promotion.


In [ ]:
# ── Promotion decision: candidate vs incumbent on the SAME test_evidence ──

# Candidate data cutoff: reject before registration when any split's data
# extends beyond the current UTC date.

from src.features.columns import FEATURE_COLS

with open(f"{input_dir}/split_meta.json") as f:
    split_meta = json.load(f)

check_candidate_cutoff(date.fromisoformat(split_meta["max_match_date"]), datetime.now(UTC).date())

feature_cols = [str(col) for col in FEATURE_COLS]
feature_cols_json = json.dumps(feature_cols, separators=(",", ":"))
feature_cols_hash = hashlib.sha256(feature_cols_json.encode()).hexdigest()

print(f"Feature contract: {len(feature_cols)} columns ({feature_cols_hash[:12]})")

champion_feature_hash = None
contract_changed = True
if champion is not None:
    contract_changed, champion_feature_hash = resolve_champion_feature_contract(
        client, champion, feature_cols, feature_cols_hash
    )


# Probability-first gate on CALIBRATED candidate probabilities: strictly lower
# calibrated candidate test log loss and ROC-AUC no more than
# PROMOTION_TOLERANCE below the incumbent's recorded log loss. Printed
# here as diagnostics; decide_promotion below makes the call.

if prod_metrics is not None:
    print("Promotion gate diagnostics (candidate vs champion):")

    print(
        f"  log_loss  candidate={cand_metrics['log_loss']:.4f} champion={prod_metrics['log_loss']:.4f} "
        f"(must be strictly lower to promote)"
    )

    print(
        f"  roc_auc   candidate={cand_metrics['roc_auc']:.4f} champion={prod_metrics['roc_auc']:.4f} "
        f"(may trail by at most {PROMOTION_TOLERANCE})"
    )

else:
    print("No production model found — first promotion.")


# Idempotency guard kept: no re-promotion when @champion already points at this run.

champion_run_id = getattr(champion, "run_id", None)

promoted = decide_promotion(
    cand_metrics=cand_metrics,
    prod_metrics=prod_metrics,
    champion_run_id=champion_run_id,
    candidate_run_id=manifest["candidate_run_id"],
    force=force_promote,
    champion_feature_hash=champion_feature_hash,
    candidate_feature_hash=feature_cols_hash,
)
if force_promote:
    promotion_reason = "force-promote requested"
elif contract_changed:
    promotion_reason = "FEATURE_COLS contract changed"
elif champion_run_id is not None and str(champion_run_id) == str(manifest["candidate_run_id"]):
    promotion_reason = "candidate is already the champion"
elif prod_metrics is None:
    promotion_reason = "no scored incumbent (first promotion)"
elif not promoted:
    # Rejected by the metric gate: explain which direction fell outside tolerance.
    if (
        cand_metrics["log_loss"] >= prod_metrics["log_loss"]
        and cand_metrics["roc_auc"] <= prod_metrics["roc_auc"]
    ):
        promotion_reason = (
            f"candidate log loss {cand_metrics['log_loss']:.4f} is not lower than champion "
            f"{prod_metrics['log_loss']:.4f} and ROC-AUC {cand_metrics['roc_auc']:.4f} does "
            f"not exceed champion {prod_metrics['roc_auc']:.4f}"
        )
    elif cand_metrics["log_loss"] >= prod_metrics["log_loss"]:
        promotion_reason = (
            f"candidate log loss {cand_metrics['log_loss']:.4f} is not lower than champion "
            f"{prod_metrics['log_loss']:.4f} (ROC-AUC improved but log loss exceeds tolerance)"
        )
    else:
        promotion_reason = (
            f"candidate ROC-AUC {cand_metrics['roc_auc']:.4f} trails champion "
            f"{prod_metrics['roc_auc']:.4f} by more than {PROMOTION_TOLERANCE:.4f} "
            f"(log loss improved but ROC-AUC exceeds tolerance)"
        )
else:
    promotion_reason = "candidate passed the log-loss and ROC-AUC gate"
print(
    f"Promotion decision: {'PROMOTED' if promoted else 'SKIPPED'} — {promotion_reason}; "
    f"candidate run={manifest['candidate_run_id']}"
)


if force_promote:
    print(">>> FORCE-PROMOTE: bypassing the metric gate and registering regardless")

elif contract_changed:
    print(
        ">>> PROMOTE: FEATURE_COLS contract changed from the champion — "
        "promoting regardless of the metric gate"
    )

elif champion_run_id is not None and str(champion_run_id) == str(manifest["candidate_run_id"]):
    print(">>> SKIP: production already points at this candidate run")

elif promoted:
    print(
        ">>> PROMOTE: candidate passed the log-loss/ROC-AUC gate (or first promotion/contract change)"
    )

else:
    print(">>> SKIP: production is still better")

In [ ]:
# ── Log decision; promotion registers only, deployment is manual ──
mlflow.set_experiment("model-evaluation")
mlflow.set_experiment_tag("pipeline", "evaluate")
with mlflow.start_run(run_name="evaluate-promotion-gate", tags={"pipeline": "evaluate"}):
    mlflow.log_metrics({f"candidate_{m}": float(cand_metrics[m]) for m in METRIC_NAMES})
    if prod_metrics is not None:
        mlflow.log_metrics({f"production_{m}": float(prod_metrics[m]) for m in METRIC_NAMES})
        mlflow.log_metrics(
            {f"delta_{m}": float(cand_metrics[m] - prod_metrics[m]) for m in METRIC_NAMES}
        )
    mlflow.log_metric("promoted", promoted)
    mlflow.log_param("candidate_run_id", manifest["candidate_run_id"])
    mlflow.log_param("promotion_reason", promotion_reason)

    eval_run = mlflow.active_run()
    assert eval_run is not None
    eval_run_id = eval_run.info.run_id

    if promoted:
        # Pin the fitted calibration temperature on this promotion: log the
        # persisted artifact under the eval run and freeze its URI+sha256 so
        # deploy downloads and hash-verifies exactly these bytes.
        cal_json = CALIBRATION_STATE.read_bytes()
        cal_hash = hashlib.sha256(cal_json).hexdigest()
        mlflow.log_artifact(str(CALIBRATION_STATE))
        cal_uri = f"runs:/{eval_run_id}/calibration_t.json"
        # Register every model together only on promotion: the ensemble plus
        # all three bases (02 logged them as runs but never registered).
        registered_pins = {}
        for name, pin in manifest["base_pins"].items():
            base_mv = mlflow.register_model(pin["model_uri"], pin["registered_model_name"])
            client.set_registered_model_tag(pin["registered_model_name"], "pipeline", "promote")
            client.set_model_version_tag(
                pin["registered_model_name"], str(base_mv.version), "pipeline", "promote"
            )
            client.set_model_version_tag(
                pin["registered_model_name"],
                str(base_mv.version),
                "pipeline_source_run_id",
                pin["run_id"],
            )
            registered_pins[name] = {
                **pin,
                "version": str(base_mv.version),
                "model_uri": f"models:/{pin['registered_model_name']}/{base_mv.version}",
            }
            print(f"Registered base {name} as {pin['registered_model_name']} v{base_mv.version}")

        mv = mlflow.register_model(manifest["model_uri"], production_model_name)
        client.set_registered_model_tag(production_model_name, "pipeline", "promote")
        # Freeze exact lineage on the promoted version BEFORE @champion is
        # assigned: these tags are the only authority deploy resolves from.
        lineage_tags = build_lineage_tags(registered_pins, manifest["aux_pins"])
        # Record each tuning notebook's winning framework and split constants on the champion lineage
        # so deploy/serving resolve them without re-detecting the saved model object.
        for _name, _pin in registered_pins.items():
            _bp = client.get_run(_pin["run_id"]).data.params
            if "framework" in _bp:
                _framework = _bp["framework"]
                if _name == "linear":
                    _framework = normalize_linear_framework(_framework).value
                elif _name == "gbdt":
                    _framework = normalize_gbdt_framework(_framework).value
                lineage_tags[f"base_{_name}_framework"] = _framework
            for _sp in (
                "train_fraction",
                "val_fraction",
                "test_fraction",
                "cv_folds",
                "random_state",
                "selection_split",
                "grouping",
            ):
                if _sp in _bp:
                    lineage_tags[f"base_{_name}_{_sp}"] = _bp[_sp]
        # Immutable selection metrics + selection artifact IDs.
        for _name, _pin in registered_pins.items():
            _bp = client.get_run(_pin["run_id"]).data.params
            for _sm in ("best_val_log_loss", "best_val_roc_auc"):
                if _sm in _bp:
                    lineage_tags[f"base_{_name}_selection_{_sm}"] = _bp[_sm]
            _prefix = f"{_name}_"
            for _sk, _sv in _pin.items():
                if _sk.startswith(_prefix) and _sk.endswith("_parquet_sha256"):
                    lineage_tags[f"base_{_name}_{_sk[len(_prefix) :]}"] = _sv
        # Pin the split and recency configuration used to build this candidate.
        lineage_tags[MIN_TRAINING_DATE_KEY] = str(manifest["min_training_date"])
        lineage_tags[TRAIN_FRACTION_KEY] = str(manifest["train_fraction"])
        lineage_tags[VAL_FRACTION_KEY] = str(manifest["val_fraction"])
        lineage_tags[TEST_FRACTION_KEY] = str(manifest["test_fraction"])
        lineage_tags[RECENCY_HALF_LIFE_KEY] = str(RECENCY_HALF_LIFE_DAYS)
        lineage_tags[RECENCY_CUTOFF_KEY] = (
            manifest.get("recency_cutoff_date") or split_meta["max_match_date"]
        )
        lineage_tags[FEATURE_COLS_TAG] = feature_cols_json
        lineage_tags[FEATURE_COLS_HASH_TAG] = feature_cols_hash
        lineage_tags["pipeline"] = "promote"
        lineage_tags["pipeline_source_run_id"] = eval_run_id
        lineage_tags[CALIBRATION_URI_TAG] = cal_uri
        lineage_tags[CALIBRATION_HASH_TAG] = cal_hash
        for key, value in lineage_tags.items():
            client.set_model_version_tag(production_model_name, str(mv.version), key, value)
        # Pin the training-data watermark (latest match date present in the
        # training splits) so drift checks cut off on data, not registration time.
        client.set_model_version_tag(
            production_model_name,
            str(mv.version),
            TRAIN_DATA_MAX_DATE_KEY,
            split_meta["max_match_date"],
        )
        # Pin the champion's 4 gate metrics so drift compares current
        # performance against this promotion-time reference.
        for m in METRIC_NAMES:
            client.set_model_version_tag(
                production_model_name,
                str(mv.version),
                f"{METRIC_PREFIX}{m}",
                str(float(cand_metrics[m])),
            )
        # Pin the champion's reference curves (ROC/PR/calibration) on this
        # promotion so every later evaluation overlays the incumbent without
        # loading it. Reference computed on this promotion-time eval split.
        from src.evaluate.curves import compute_curve_data, serialize_curve_data
        import tempfile as _tmp

        _champion_curve = compute_curve_data(y_eval, y_proba)
        _curve_bytes, _curve_hash = serialize_curve_data(_champion_curve)
        # Log under a fixed name at the run root so the tagged URI points at the
        # exact artifact the champion version later resolves.
        _curve_path = f"{_tmp.gettempdir()}/{CHAMPION_CURVE_ARTIFACT}"
        with open(_curve_path, "wb") as _cf:
            _cf.write(_curve_bytes)
        mlflow.log_artifact(_curve_path)
        lineage_tags[CHAMPION_CURVE_URI_TAG] = f"runs:/{eval_run_id}/{CHAMPION_CURVE_ARTIFACT}"
        lineage_tags[CHAMPION_CURVE_HASH_TAG] = _curve_hash
        eval_tags = {
            EVAL_SPLIT_SIZE_KEY: str(len(test_eval)),
            EVAL_MAX_DATE_KEY: str(pd.to_datetime(chosen_info["match_date"]).max().date()),
        }
        for key, value in eval_tags.items():
            client.set_model_version_tag(production_model_name, str(mv.version), key, value)
        # Assign @champion directly to the promoted candidate version. Promotion
        # now pins the champion here, so a failed promotion leaves the prior
        # champion untouched.
        champion_curve_uri = lineage_tags[CHAMPION_CURVE_URI_TAG]
        champion_curve_hash = lineage_tags[CHAMPION_CURVE_HASH_TAG]
        client.set_registered_model_alias(PRODUCTION_MODEL, CHAMPION_ALIAS, str(mv.version))
        manifest["selected_production_version"] = str(mv.version)
        manifest["selected_run_id"] = eval_run_id
        manifest["candidate_metrics"] = {m: float(cand_metrics[m]) for m in METRIC_NAMES}
        manifest["champion_curve_uri"] = champion_curve_uri
        manifest["champion_curve_hash"] = champion_curve_hash
        manifest["calibration_uri"] = cal_uri
        manifest["calibration_hash"] = cal_hash
        manifest["train_data_max_date"] = split_meta["max_match_date"]
        manifest["eval_split_size"] = len(test_eval)
        manifest["eval_max_date"] = str(pd.to_datetime(chosen_info["match_date"]).max().date())
        with open(candidate_manifest, "w") as _f:
            json.dump(manifest, _f, indent=2)
        print(
            f"Registered candidate {manifest['model_uri']} as '{production_model_name}' "
            f"v{mv.version} and assigned @{CHAMPION_ALIAS}"
        )
        print(
            f"Selected candidate v{mv.version} promoted as champion "
            f"({production_model_name}@{CHAMPION_ALIAS})"
        )

        # Optional manual deploy hook — off by default, never wired to a build.
        if rebuild_cmd:
            print(f"Executing manual deploy hook: {rebuild_cmd}")
            result = subprocess.run(rebuild_cmd, shell=True)
            print(f"Deploy hook finished with return code {result.returncode}")
    else:
        print("No promotion — skipped registration. Deployment (if wanted) stays manual.")
    if not promoted:
        current = resolve_champion(client)
        print(
            f"Champion unchanged: {production_model_name} "
            f"v{getattr(current, 'version', 'none')} (candidate was not registered)"
        )
    print(
        f"EVALUATION COMPLETE: candidate {'PROMOTED' if promoted else 'SKIPPED'}; "
        f"reason={promotion_reason}"
    )

## Report — always runs

Everything below renders unconditionally (the promotion outcome does not gate
it) and is logged to the evaluation run: ROC curves (three bases + candidate + champion
when pinned), PR and calibration curves for the candidate (plus the pinned champion
overlay when available), the candidate's confusion matrix,
and best-effort SHAP attribution on the GBDT candidate.


In [ ]:
# ── Resolve the incumbent champion's pinned reference curves ──
from src.evaluate.curves import resolve_champion_curves

champion_curve = resolve_champion_curves(client, champion)
CALIBRATION_PLOTS.mkdir(parents=True, exist_ok=True)

# ── ROC curves: 3 base models + candidate + champion (pinned) ──
with mlflow.start_run(run_id=eval_run_id):
    fig, ax = plt.subplots(figsize=report_figsize)
    for name in STACK_ORDER:
        p = evidence_to_probability(test_evidence[name])
        fpr, tpr, _ = roc_curve(y_eval, p)
        ax.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_eval, p):.3f})", lw=1.5)
    fpr, tpr, _ = roc_curve(y_eval, y_proba)
    ax.plot(
        fpr, tpr, label=f"candidate (AUC {roc_auc_score(y_eval, y_proba):.3f})", lw=2.5, ls="--"
    )
    if champion_curve is not None:
        ax.plot(
            champion_curve.roc["fpr"],
            champion_curve.roc["tpr"],
            label=f"champion (AUC {champion_curve.roc['auc']:.3f})",
            lw=2.0,
            color="black",
        )
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("ROC curves — base models, candidate, champion")
    ax.legend(loc="lower right")
    fig.tight_layout()
    mlflow.log_figure(fig, "figures/roc_curves.png")
    fig.savefig(CALIBRATION_PLOTS / "roc_curves.png", dpi=report_dpi)
    display(fig)
    plt.close(fig)

In [ ]:
# ── PR curves: candidate + champion (pinned) ──
with mlflow.start_run(run_id=eval_run_id):
    fig, ax = plt.subplots(figsize=report_figsize)
    precision, recall, _ = precision_recall_curve(y_eval, y_proba)
    ax.plot(
        recall,
        precision,
        label=f"candidate (AP {average_precision_score(y_eval, y_proba):.3f})",
        lw=2.5,
    )
    if champion_curve is not None:
        ax.plot(
            champion_curve.pr["recall"],
            champion_curve.pr["precision"],
            label=f"champion (AP {champion_curve.pr['ap']:.3f})",
            lw=2.0,
            color="black",
        )
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-recall curve — candidate, champion")
    ax.legend(loc="lower left")
    fig.tight_layout()
    mlflow.log_figure(fig, "figures/pr_curves.png")
    fig.savefig(CALIBRATION_PLOTS / "pr_curves.png", dpi=report_dpi)
    display(fig)
    plt.close(fig)

In [ ]:
# ── Calibration (reliability) curves: calibrated candidate, raw, champion ──
with mlflow.start_run(run_id=eval_run_id):
    fig, ax = plt.subplots(figsize=report_figsize)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="perfect")
    prob_true, prob_pred = calibration_curve(y_eval, y_proba, n_bins=calibration_n_bins)
    ax.plot(
        prob_pred,
        prob_true,
        marker="o",
        label=f"calibrated candidate (Brier {brier_score_loss(y_eval, y_proba):.4f})",
    )
    # Raw candidate kept as a faint diagnostic (temperature changes only the
    # probability scale, never the ranking).
    prob_true, prob_pred = calibration_curve(y_eval, y_proba_raw, n_bins=calibration_n_bins)
    ax.plot(
        prob_pred,
        prob_true,
        marker=".",
        ls=":",
        color="gray",
        alpha=0.6,
        label=f"raw candidate (Brier {brier_score_loss(y_eval, y_proba_raw):.4f})",
    )
    if champion_curve is not None:
        ax.plot(
            champion_curve.calibration["prob_pred"],
            champion_curve.calibration["prob_true"],
            marker="s",
            color="black",
            label=f"champion (Brier {champion_curve.calibration['brier']:.4f})",
        )
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.set_title("Calibration curve — calibrated candidate (raw candidate diagnostic)")
    ax.legend(loc="lower right")
    fig.tight_layout()
    mlflow.log_figure(fig, "figures/calibration_curves.png")
    fig.savefig(CALIBRATION_PLOTS / "calibration_curves.png", dpi=report_dpi)
    display(fig)
    plt.close(fig)

In [ ]:
# ── Confusion matrix: candidate ──
with mlflow.start_run(run_id=eval_run_id):
    models_to_plot = [("candidate", y_pred)]
    fig, axes = plt.subplots(1, len(models_to_plot), figsize=(6 * len(models_to_plot), 5))
    if len(models_to_plot) == 1:
        axes = [axes]
    for ax, (title, pred) in zip(axes, models_to_plot, strict=False):
        sns.heatmap(confusion_matrix(y_eval, pred), annot=True, fmt="d", cmap="Blues", ax=ax)
        ax.set_title(f"{title} confusion matrix")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
    fig.tight_layout()
    mlflow.log_figure(fig, "figures/confusion_matrices.png")
    fig.savefig(CALIBRATION_PLOTS / "confusion_matrices.png", dpi=report_dpi)
    display(fig)
    plt.close(fig)

## SHAP feature attribution (best-effort)

`shap.TreeExplainer` on the pinned GBDT candidate (loaded from MLflow, never a
latest lookup) over a `FEATURE_COLS` subsample of `X_test`. Try/except: a GBDT
the flavor cannot handle only skips this section. Runs on the raw directional
test rows (two per match), not the evidence — attribution is per model input
row, so it is unchanged by the evidence conversion.


In [ ]:
# ── SHAP: TreeExplainer on the pinned GBDT candidate (best-effort, never blocks the report) ──
# Slow/fragile on some GBDT packages — any failure logs a warning and the
# rest of the report still renders.
try:
    import shap

    from src.features.columns import FEATURE_COLS as feature_cols

    X_shap = pd.read_parquet(f"{input_dir}/X_test.parquet").reset_index(drop=True)
    cols = [c for c in feature_cols if c in X_shap.columns]
    X_shap = X_shap[cols]
    sample = X_shap.sample(n=min(shap_sample_size, len(X_shap)), random_state=random_state)
    gbdt = mlflow.sklearn.load_model(manifest["base_pins"]["gbdt"]["model_uri"])
    assert gbdt is not None
    explainer = shap.TreeExplainer(gbdt)
    shap_values = explainer(sample)
    print(
        f"SHAP computed on {len(sample)} rows x {len(cols)} features "
        f"({len(feature_cols)} declared in FEATURE_COLS)"
    )

    with mlflow.start_run(run_id=eval_run_id):
        shap.plots.beeswarm(
            shap_values,
            max_display=len(cols),
            order=shap_values.abs.mean(0),
            show=False,
        )
        plt.title("SHAP beeswarm — pinned gbdt")
        mlflow.log_figure(plt.gcf(), "figures/shap_beeswarm.png")
        plt.gcf().savefig(
            CALIBRATION_PLOTS / "shap_beeswarm.png",
            dpi=report_dpi,
            bbox_inches="tight",
            pad_inches=0.1,
        )
        display(plt.gcf())
        plt.close(plt.gcf())

        gbdt_importance = (
            pd.DataFrame({"feature": cols, "importance": shap_values.abs.mean(0).values})
            .sort_values("importance", ascending=False)
            .reset_index(drop=True)
        )
        gbdt_importance.to_csv(CALIBRATION_PLOTS / "gbdt_feature_importance.csv", index=False)
        print("GBDT feature importance (mean absolute SHAP):")
        display(gbdt_importance)
        ranked = gbdt_importance.head(20).sort_values("importance", ascending=True)
        fig, ax = plt.subplots(figsize=(10, max(6, 0.3 * len(ranked))))
        ax.barh(ranked["feature"], ranked["importance"])
        ax.set_title("GBDT feature importance - mean absolute SHAP")
        ax.set_xlabel("Mean absolute SHAP value")
        fig.tight_layout()
        mlflow.log_artifact(str(CALIBRATION_PLOTS / "gbdt_feature_importance.csv"))
        mlflow.log_figure(fig, "figures/gbdt_feature_importance.png")
        fig.savefig(
            CALIBRATION_PLOTS / "gbdt_feature_importance.png", dpi=report_dpi, bbox_inches="tight"
        )
        display(fig)
        plt.close(fig)
except Exception as e:
    print(f"WARNING: SHAP analysis failed and was skipped: {e}")

## Linear feature importance (best-effort)

Logistic regression uses absolute standardized coefficients. Gaussian Naive
Bayes uses standardized class-mean separation. The table and chart are sorted
with the most influential features first.


In [ ]:
try:
    from src.features.columns import FEATURE_COLS as feature_cols

    linear = mlflow.sklearn.load_model(manifest["base_pins"]["linear"]["model_uri"])
    assert linear is not None
    if hasattr(linear, "coef_"):
        signed = np.asarray(linear.coef_).reshape(-1)
        importance = np.abs(signed)
        metric = "absolute standardized coefficient"
    elif hasattr(linear, "theta_") and hasattr(linear, "var_"):
        signed = np.asarray(linear.theta_[1] - linear.theta_[0]).reshape(-1)
        pooled_std = np.sqrt(np.maximum(np.asarray(linear.var_).mean(axis=0), np.finfo(float).eps))
        importance = np.abs(signed) / pooled_std
        metric = "standardized class-mean separation"
    else:
        raise TypeError(f"unsupported linear model: {type(linear).__name__}")  # noqa: TRY301

    linear_importance = (
        pd.DataFrame({"feature": feature_cols, "importance": importance, "signed": signed})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    linear_importance.to_csv(CALIBRATION_PLOTS / "linear_feature_importance.csv", index=False)
    print(f"Linear feature importance ({metric}):")
    display(linear_importance)

    with mlflow.start_run(run_id=eval_run_id):
        mlflow.log_artifact(str(CALIBRATION_PLOTS / "linear_feature_importance.csv"))
        ranked = linear_importance.sort_values("importance", ascending=True)
        fig, ax = plt.subplots(figsize=(10, max(6, 0.3 * len(ranked))))
        ax.barh(ranked["feature"], ranked["importance"])
        ax.set_title(f"Linear feature importance - {metric}")
        ax.set_xlabel("Absolute influence")
        fig.tight_layout()
        mlflow.log_figure(fig, "figures/linear_feature_importance.png")
        fig.savefig(
            CALIBRATION_PLOTS / "linear_feature_importance.png", dpi=report_dpi, bbox_inches="tight"
        )
        display(fig)
        plt.close(fig)
except Exception as e:
    print(f"WARNING: linear feature importance failed and was skipped: {e}")